# Scenario-Based Stress Testing

This notebook quantifies the sensitivity of CVA, DVA, and BCVA to adverse market scenarios, extending the regime-conditional base case estimates from Notebook 5.

Stress testing is a core component of counterparty credit risk management. While the base-case XVA estimates reflect average market conditions within each historical regime, a CMR framework must also quantify how valuation adjustments behave under tail events that may exceed historically observed market dynamics.

## Scenarios

**Scenario 1 — Credit Spread Widening**
- BBB OAS: +100 bps
- IG OAS: +75 bps
- Rates: unchanged
- Exposure: unchanged

This scenario isolates the pure credit deterioration channel of XVA sensitivity. Only hazard rates are re-calibrated from stressed OAS levels, while discount curves and exposure dynamics remain unchanged.

**Scenario 2 — Market Stress / Volatility Shock**
- BBB OAS: +150 bps
- IG OAS: +100 bps
- Hull-White σ: × 1.5
- Discount curve: unchanged
- Exposure: re-simulated under stressed $\sigma$

This scenario captures acute market dislocation where deteriorating credit conditions occur simultaneously with elevated rate volatility, amplifying both exposure uncertainty and default risk.

**Scenario 3 — Stagflation / Rates + Credit Shock**
- Yield curve: +100 bps parallel shift
- BBB OAS: +150 bps
- IG OAS: +100 bps
- Discount curve: re-computed from shocked yields
- Rate paths: re-simulated under shocked $\theta_t$
- Exposure: re-simulated

This is the most comprehensive stress scenario, simultaneously affecting discounting, exposure dynamics, and default probabilities. The scenario is intended to approximate a persistent high-rate environment accompanied by deteriorating credit quality.

## Methodology

Each scenario is applied as an overlay on top of the four regime-conditional base cases from Notebook 5. For each regime × scenario combination, we:

1. Apply the scenario shocks to relevant inputs
2. Recompute stressed hazard rates where required
3. Recompute stressed discount curves where required
4. Re-simulate exposure profiles where required
5. Recompute UCVA, UDVA, and BCVA
6. Compute $\delta$ XVA = stressed XVA − base XVA

This produces a $4 * 3$ regime-scenario matrix of stress impacts, providing a comprehensive view of CVA sensitivity across both market environments and shock types.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.interpolate import CubicSpline
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

np.random.seed(926)

# Load inputs
df = pd.read_csv('../data/processed/main_dataset.csv',index_col='date', parse_dates=True)
regimes = pd.read_csv('../data/processed/regime_labels.csv',index_col='date', parse_dates=True)
hazard_rates = pd.read_csv('../data/processed/regime_hazard_rates.csv')
xva_base = pd.read_csv('../data/processed/xva_results.csv')
exposure_df = pd.read_csv('../data/processed/exposure_profiles.csv')

df = df.join(regimes, how='inner')

# Parameters 
NOTIONAL = 100_000_000
LGD = 0.60
N_PATHS = 1000
N_STEPS = 60
DT = 1/12
CP_RATING  = 'BBB'
OWN_RATING = 'IG'

REGIME_ORDER = [
    'Risk-On',
    'Normalization',
    'Event-Driven Stress',
    'Macro Tightening',
]
REGIME_COLORS = {
    'Risk-On': 'steelblue',
    'Normalization': 'mediumseagreen',
    'Event-Driven Stress': 'darkorange',
    'Macro Tightening': 'crimson',
}
SCENARIO_COLORS = {
    'Base': 'dimgray',
    'Scenario 1\nCredit Spread': 'steelblue',
    'Scenario 2\nMarket Stress': 'darkorange',
    'Scenario 3\nStagflation': 'crimson',
}

YIELD_COLS = ['dgs1mo', 'dgs3mo', 'dgs6mo',
                   'dgs1', 'dgs2', 'dgs3',
                   'dgs5', 'dgs10']
YIELD_TENORS_MO = np.array([1, 3, 6, 12,
                              24, 36, 60, 120])
SIM_TENORS_MO = np.arange(1, 61)

# Reconstruct functions
def calibrate_vasicek(r_series, dt=1/252):
    r = np.array(r_series)
    dr = np.diff(r)
    r_lag  = r[:-1]
    X = np.column_stack([np.ones(len(r_lag)), r_lag])
    coeffs = np.linalg.lstsq(X, dr, rcond=None)[0]
    a, b = coeffs
    kappa = max(-b / dt, 0.001)
    theta = a / (kappa * dt)
    resid = dr - (a + b * r_lag)
    sigma = max(np.std(resid) / np.sqrt(dt), 0.0001)
    return kappa, sigma, theta

def compute_hw_theta(kappa, sigma,
                     yield_curve_pct,
                     sim_tenors_mo,
                     cs_spline=None):
    t = sim_tenors_mo / 12
    if cs_spline is not None:
        r_t = cs_spline(sim_tenors_mo) / 100
        dr_dt = cs_spline.derivative()(sim_tenors_mo) / 100 * 12
        d2r_dt2 = cs_spline.derivative(2)(sim_tenors_mo) / 100 * 144
        f_t = r_t + t * dr_dt
        df_dt = dr_dt + t * d2r_dt2
    else:
        r = yield_curve_pct / 100
        P = np.exp(-r * t)
        ln_P = np.log(P)
        dt_yr = 1/12
        f_t = np.zeros(len(t))
        f_t[0] = -(ln_P[1]-ln_P[0])/dt_yr
        f_t[-1] = -(ln_P[-1]-ln_P[-2])/dt_yr
        f_t[1:-1] = -(ln_P[2:]-ln_P[:-2])/(
                          2*dt_yr)
        df_dt = np.zeros(len(t))
        df_dt[0] = (f_t[1]-f_t[0])/dt_yr
        df_dt[-1] = (f_t[-1]-f_t[-2])/dt_yr
        df_dt[1:-1] = (f_t[2:]-f_t[:-2])/(2*dt_yr)
    convexity = (sigma**2/(2*kappa**2))*(1 - np.exp(-2*kappa*t))
    theta_t = (1/kappa)*df_dt + f_t + convexity
    return theta_t, f_t

def simulate_hull_white(kappa, sigma,
                        theta_t, r0,
                        n_paths, n_steps, dt):
    paths = np.zeros((n_paths, n_steps+1))
    paths[:, 0] = r0
    for t in range(n_steps):
        eps = np.random.randn(n_paths)
        drift = kappa*(theta_t[t]-paths[:,t])*dt
        diff = sigma * np.sqrt(dt) * eps
        paths[:, t+1] = paths[:, t]+drift+diff
    return paths

def compute_irs_value(paths, K, dt=1/12):
    n_paths, n_steps_1 = paths.shape
    n_steps = n_steps_1 - 1
    values  = np.zeros((n_paths, n_steps))
    for t in range(n_steps):
        remaining = paths[:, t+1:]
        n_rem     = remaining.shape[1]
        if n_rem == 0:
            continue
        cum_rates = np.cumsum(
            remaining * dt, axis=1
        )
        df_path = np.exp(-cum_rates)
        net_cf  = (remaining-K)*dt*NOTIONAL
        values[:, t] = np.sum(
            df_path * net_cf, axis=1
        )
    return values

def compute_par_swap_rate(yield_curve_dec,
                           dt=1/12,
                           n_steps=60):
    t_years = np.arange(1, n_steps+1) / 12
    DF = np.exp(-yield_curve_dec * t_years)
    K = (1-DF[-1]) / (np.sum(DF)*dt)
    return K

def compute_xva(epe_profile, ene_profile,
                df_curve, lam_cp, lam_own,
                lgd=0.60, dt=1/12):
    n_steps = len(epe_profile)
    t_mo = np.arange(1, n_steps+1) * dt
    S_cp = np.exp(-lam_cp  * t_mo)
    S_own = np.exp(-lam_own * t_mo)
    S_cp_lag = np.concatenate([[1.0], S_cp[:-1]])
    S_own_lag = np.concatenate([[1.0], S_own[:-1]])
    q_cp = S_cp_lag  - S_cp
    q_own = S_own_lag - S_own

    ucva = -lgd * np.sum(df_curve * epe_profile * q_cp)
    udva = -lgd * np.sum(df_curve * ene_profile * q_own)

    bcva_cva = -lgd * np.sum(df_curve * epe_profile * q_cp * S_own)
    bcva_dva = -lgd * np.sum(df_curve * ene_profile * q_own * S_cp)
    bcva = bcva_cva + bcva_dva
    return ucva, udva, bcva

# Reconstruct regime yield curves
regime_yield_curves = {}
regime_discount_curves = {}
regime_splines = {}
vasicek_params = {}
hw_params = {}

for regime in REGIME_ORDER:
    mask = df['regime_name'] == regime
    mean_yields = df.loc[mask, YIELD_COLS].mean()

    cs = CubicSpline(
        YIELD_TENORS_MO,
        mean_yields.values,
        bc_type='natural'
    )
    regime_splines[regime] = cs

    yc = np.maximum(cs(SIM_TENORS_MO), 0.001)
    regime_yield_curves[regime] = yc

    t_yr = SIM_TENORS_MO / 12
    regime_discount_curves[regime] = np.exp(
        -yc / 100 * t_yr
    )

    # Vasicek calibration
    r_data = df.loc[mask, 'dgs1mo'].dropna()/100
    kappa, sigma, theta = calibrate_vasicek(
        r_data, dt=1/252
    )
    vasicek_params[regime] = {
        'kappa': kappa,
        'sigma': sigma,
        'theta': theta,
    }

    # HW theta
    theta_t, _ = compute_hw_theta(
        kappa, sigma, yc, SIM_TENORS_MO,
        cs_spline=cs
    )
    hw_params[regime] = {
        'kappa': kappa,
        'sigma': sigma,
        'theta_t': theta_t,
    }

# Reconstruct base EPE/ENE 
epe_profiles = {}
ene_profiles = {}

for regime in REGIME_ORDER:
    subset = exposure_df[
        exposure_df['regime'] == regime
    ].sort_values('tenor')
    epe_profiles[regime] = subset['epe'].values
    ene_profiles[regime] = subset['ene'].values

# Unified K
full_yc_dec = np.array([
    df[col].mean()/100 for col in YIELD_COLS
])
cs_full = CubicSpline(
    YIELD_TENORS_MO,
    full_yc_dec * 100,
    bc_type='natural'
)
full_yc_interp = np.maximum(
    cs_full(SIM_TENORS_MO)/100, 0.001
)
K_UNIFIED = compute_par_swap_rate(full_yc_interp)

#  Base hazard rates
base_lambda = {}
for regime in REGIME_ORDER:
    base_lambda[regime] = {
        'cp':  hazard_rates.loc[
            (hazard_rates['regime'] == regime) &
            (hazard_rates['rating'] == CP_RATING),
            'lambda'
        ].values[0],
        'own': hazard_rates.loc[
            (hazard_rates['regime'] == regime) &
            (hazard_rates['rating'] == OWN_RATING),
            'lambda'
        ].values[0],
    }

print("Setup complete")
print(f"\nUnified K: {K_UNIFIED*100:.4f}%")
print(f"\nBase hazard rates:")
print(f"{'Regime':<22} {'lambda_cp (BBB)':>12} "
      f"{'lambda_own (IG)':>12}")
print("-"*50)
for regime in REGIME_ORDER:
    print(f"{regime:<22} "
          f"{base_lambda[regime]['cp']:>12.6f} "
          f"{base_lambda[regime]['own']:>12.6f}")

print(f"\nBase XVA:")
print(xva_base[['regime','ucva',
                'udva','bcva']].to_string(
    index=False
))

Setup complete

Unified K: 3.4595%

Base hazard rates:
Regime                 lambda_cp (BBB) lambda_own (IG)
--------------------------------------------------
Risk-On                    0.017675     0.014130
Normalization              0.020339     0.016372
Event-Driven Stress        0.023490     0.019030
Macro Tightening           0.029689     0.023995

Base XVA:
             regime          ucva         udva          bcva
            Risk-On -54572.107204 28560.447711 -25286.279529
      Normalization -10616.119231 27132.512463  15831.863414
Event-Driven Stress -58591.596191 36105.885760 -21910.650122
   Macro Tightening -17011.917693 53106.525667  33722.595941
